# RAG com Ollama (local) — notebook adaptado

Versão adaptada para rodar **localmente com Ollama**, substituindo a etapa de geração que antes usava OpenAI.

## O que este notebook faz
- lê arquivos `.txt`, `.md`, `.csv`, `.jsonl`, `.pdf`
- faz chunking com overlap
- gera embeddings com `sentence-transformers`
- indexa com `FAISS`
- usa **Ollama** para responder localmente
- mostra fontes usadas
- permite testar com dataset fake antes de usar seus dados

## Arquitetura
- **Embeddings**: `sentence-transformers/all-MiniLM-L6-v2`
- **Vector store**: `FAISS`
- **LLM local**: `llama3.1` via Ollama

## 1) Antes de começar

Garanta que o Ollama esteja instalado e rodando.

### Instalação
```bash
brew install ollama
```

### Subir o serviço
```bash
ollama serve
```

### Baixar um modelo
```bash
ollama pull llama3.1
```

Se quiser usar outro modelo depois, basta trocar em `config.llm_model`.

## 2) Instalação das libs Python

In [1]:
# !pip install -U pandas numpy faiss-cpu sentence-transformers scikit-learn pypdf requests tqdm

  Using cached numpy-2.4.4-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached faiss_cpu-1.13.2-cp310-abi3-macosx_14_0_arm64.whl.metadata (7.6 kB)
Using cached numpy-2.4.4-cp311-cp311-macosx_14_0_arm64.whl (5.5 MB)
Using cached faiss_cpu-1.13.2-cp310-abi3-macosx_14_0_arm64.whl (3.5 MB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: faiss-cpu0m━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [numpy]
    Found existing installation: faiss-cpu 1.8.0━━━━━━━━━━━━━ 2/3 [faiss-cpu]
    Uninstalling faiss-cpu-1.8.0:m╸━━━━━━━━━━━━━ 2/3 [faiss-cpu]
      Successfully uninstalled faiss-cpu-1.8.090m━━━━━━━━━━━━━ 2/3 [faiss-cpu]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [faiss-cpu]0m [faiss-cpu]


In [1]:
from __future__ import annotations

import re
import json
import time
import pickle
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

/Users/yandrade/PycharmProjects/ia_study/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3) Configuração

In [2]:
@dataclass
class RAGConfig:
    data_dir: str = "./rag_data"
    index_dir: str = "./rag_index"

    chunk_size: int = 900
    chunk_overlap: int = 180
    min_chunk_chars: int = 120

    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    normalize_embeddings: bool = True

    top_k: int = 8
    top_k_rerank: int = 4
    use_reranker: bool = False
    reranker_model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    llm_model: str = "llama3.1"
    ollama_base_url: str = "http://localhost:11434"
    temperature: float = 0.1
    max_output_tokens: int = 700

    csv_text_columns_priority: Tuple[str, ...] = (
        "text", "content", "description", "message", "body", "notes"
    )

config = RAGConfig()
config

RAGConfig(data_dir='./rag_data', index_dir='./rag_index', chunk_size=900, chunk_overlap=180, min_chunk_chars=120, embedding_model_name='sentence-transformers/all-MiniLM-L6-v2', normalize_embeddings=True, top_k=8, top_k_rerank=4, use_reranker=False, reranker_model_name='cross-encoder/ms-marco-MiniLM-L-6-v2', llm_model='llama3.1', ollama_base_url='http://localhost:11434', temperature=0.1, max_output_tokens=700, csv_text_columns_priority=('text', 'content', 'description', 'message', 'body', 'notes'))

## 4) Helpers

In [3]:
def sha1_text(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

def clean_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text).replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p

def approx_token_count(text: str) -> int:
    return max(1, int(len(text) / 4))

## 5) Loaders

In [4]:
def load_txt_or_md(path: Path) -> List[Dict[str, Any]]:
    text = clean_text(path.read_text(encoding="utf-8", errors="ignore"))
    return [{
        "doc_id": sha1_text(f"{path}:{text[:500]}"),
        "source_path": str(path),
        "source_type": path.suffix.lower().lstrip("."),
        "title": path.stem,
        "text": text,
        "metadata": {"file_name": path.name}
    }]

def detect_csv_text_column(df: pd.DataFrame, priority_cols: Tuple[str, ...]) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for p in priority_cols:
        if p.lower() in lower_map:
            return lower_map[p.lower()]

    object_cols = [c for c in df.columns if df[c].dtype == "object"]
    if not object_cols:
        return None

    scored = []
    for c in object_cols:
        series = df[c].dropna().astype(str)
        if len(series) == 0:
            continue
        avg_len = series.str.len().mean()
        uniq_ratio = series.nunique() / max(1, len(series))
        score = avg_len * 0.8 + uniq_ratio * 50
        scored.append((c, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[0][0] if scored else None

def row_to_text(row: pd.Series, text_col: str) -> str:
    parts = [f"{text_col}: {row.get(text_col, '')}"]
    for c, v in row.items():
        if c == text_col:
            continue
        if pd.isna(v):
            continue
        parts.append(f"{c}: {v}")
    return clean_text("\n".join(parts))

def load_csv(path: Path, config: RAGConfig, text_col: Optional[str] = None) -> List[Dict[str, Any]]:
    df = pd.read_csv(path)
    chosen_col = text_col or detect_csv_text_column(df, config.csv_text_columns_priority)
    if chosen_col is None:
        raise ValueError(f"Não encontrei coluna textual em {path}")

    documents = []
    for idx, row in df.iterrows():
        text = row_to_text(row, chosen_col)
        if len(text.strip()) == 0:
            continue

        metadata = {
            "file_name": path.name,
            "row_index": int(idx),
            "text_column": chosen_col,
        }
        for c in df.columns[:15]:
            val = row.get(c)
            if c == chosen_col or pd.isna(val):
                continue
            metadata[f"col_{c}"] = str(val)[:200]

        documents.append({
            "doc_id": sha1_text(f"{path}:{idx}:{text[:500]}"),
            "source_path": str(path),
            "source_type": "csv_row",
            "title": f"{path.stem} | row {idx}",
            "text": text,
            "metadata": metadata,
        })
    return documents

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    documents = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for idx, line in enumerate(f):
            if not line.strip():
                continue
            obj = json.loads(line)
            text = clean_text(json.dumps(obj, ensure_ascii=False, indent=2))
            documents.append({
                "doc_id": sha1_text(f"{path}:{idx}:{text[:500]}"),
                "source_path": str(path),
                "source_type": "jsonl",
                "title": f"{path.stem} | item {idx}",
                "text": text,
                "metadata": {"file_name": path.name, "row_index": idx},
            })
    return documents

def load_pdf(path: Path) -> List[Dict[str, Any]]:
    from pypdf import PdfReader
    reader = PdfReader(str(path))
    docs = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")
        if not text:
            continue
        docs.append({
            "doc_id": sha1_text(f"{path}:page:{page_num}:{text[:500]}"),
            "source_path": str(path),
            "source_type": "pdf_page",
            "title": f"{path.stem} | page {page_num}",
            "text": text,
            "metadata": {"file_name": path.name, "page": page_num},
        })
    return docs

def load_documents(config: RAGConfig, csv_text_overrides: Optional[Dict[str, str]] = None) -> List[Dict[str, Any]]:
    root = Path(config.data_dir)
    if not root.exists():
        print(f"Pasta {root} não existe ainda.")
        return []

    documents = []
    csv_text_overrides = csv_text_overrides or {}

    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        try:
            suffix = path.suffix.lower()
            if suffix in [".txt", ".md"]:
                documents.extend(load_txt_or_md(path))
            elif suffix == ".csv":
                forced_col = csv_text_overrides.get(path.name)
                documents.extend(load_csv(path, config, forced_col))
            elif suffix == ".jsonl":
                documents.extend(load_jsonl(path))
            elif suffix == ".pdf":
                documents.extend(load_pdf(path))
        except Exception as e:
            print(f"[WARN] Falha ao carregar {path}: {e}")

    return documents

## 6) Chunking

In [5]:
def split_text_with_overlap(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    text = clean_text(text)
    if len(text) <= chunk_size:
        return [text] if text else []

    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end]

        if end < text_len:
            candidates = [
                chunk.rfind("\n\n"),
                chunk.rfind("\n"),
                chunk.rfind(". "),
                chunk.rfind("; "),
                chunk.rfind(", "),
            ]
            best = max(candidates)
            if best > int(chunk_size * 0.6):
                end = start + best + 1
                chunk = text[start:end]

        chunk = clean_text(chunk)
        if chunk:
            chunks.append(chunk)

        if end >= text_len:
            break
        start = max(end - chunk_overlap, start + 1)

    return chunks

def build_chunks(documents: List[Dict[str, Any]], config: RAGConfig) -> List[Dict[str, Any]]:
    chunks = []
    for doc in documents:
        pieces = split_text_with_overlap(doc["text"], config.chunk_size, config.chunk_overlap)
        for i, piece in enumerate(pieces):
            if len(piece) < config.min_chunk_chars:
                continue
            chunks.append({
                "chunk_id": sha1_text(f"{doc['doc_id']}:{i}:{piece[:400]}"),
                "doc_id": doc["doc_id"],
                "source_path": doc["source_path"],
                "source_type": doc["source_type"],
                "title": doc["title"],
                "text": piece,
                "metadata": {
                    **doc.get("metadata", {}),
                    "chunk_index": i,
                    "char_len": len(piece),
                    "approx_tokens": approx_token_count(piece),
                },
            })
    return chunks

In [6]:
documents = load_documents(config)
print(f"Documents carregados: {len(documents)}")

chunks = build_chunks(documents, config)
print(f"Chunks gerados: {len(chunks)}")

if chunks:
    pd.DataFrame([{
        "title": c["title"],
        "source_type": c["source_type"],
        "chars": len(c["text"]),
        "approx_tokens": c["metadata"]["approx_tokens"],
        "source_path": c["source_path"],
    } for c in chunks[:10]])
else:
    print("Coloque seus arquivos em rag_data/ e rode de novo.")

Documents carregados: 13
Chunks gerados: 17


## 7) Embeddings + índice vetorial

In [7]:
from sentence_transformers import SentenceTransformer
import faiss

class VectorIndex:
    def __init__(self, config: RAGConfig):
        self.config = config
        self.model = SentenceTransformer(config.embedding_model_name)
        self.index = None
        self.chunk_store: List[Dict[str, Any]] = []

    def embed(self, texts: List[str], batch_size: int = 64) -> np.ndarray:
        emb = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=self.config.normalize_embeddings,
        )
        return emb.astype("float32")

    def build(self, chunks: List[Dict[str, Any]]) -> None:
        self.chunk_store = chunks
        vectors = self.embed([c["text"] for c in chunks])
        dim = vectors.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(vectors)

    def search(self, query: str, top_k: int = 5, metadata_filter: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        if self.index is None:
            raise RuntimeError("Índice ainda não foi construído.")

        q = self.embed([query])
        scores, idxs = self.index.search(q, top_k * 5 if metadata_filter else top_k)

        results = []
        for score, idx in zip(scores[0], idxs[0]):
            if idx == -1:
                continue
            chunk = self.chunk_store[int(idx)]

            if metadata_filter:
                ok = True
                for k, v in metadata_filter.items():
                    if chunk["metadata"].get(k) != v:
                        ok = False
                        break
                if not ok:
                    continue

            results.append({"score": float(score), "chunk": chunk})
            if len(results) >= top_k:
                break
        return results

    def save(self, path: str | Path) -> None:
        path = ensure_dir(path)
        faiss.write_index(self.index, str(path / "faiss.index"))
        with open(path / "chunk_store.pkl", "wb") as f:
            pickle.dump(self.chunk_store, f)
        with open(path / "config.json", "w", encoding="utf-8") as f:
            json.dump(asdict(self.config), f, ensure_ascii=False, indent=2)

    def load(self, path: str | Path) -> None:
        path = Path(path)
        self.index = faiss.read_index(str(path / "faiss.index"))
        with open(path / "chunk_store.pkl", "rb") as f:
            self.chunk_store = pickle.load(f)

In [8]:
vector_index = VectorIndex(config)

if chunks:
    vector_index.build(chunks)
    print("Índice vetorial criado com sucesso.")
else:
    print("Sem chunks ainda.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11243.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]

Índice vetorial criado com sucesso.


## 8) Re-ranking opcional

In [9]:
class OptionalReranker:
    def __init__(self, model_name: str):
        from sentence_transformers import CrossEncoder
        self.model = CrossEncoder(model_name)

    def rerank(self, query: str, retrieved: List[Dict[str, Any]], top_k: int) -> List[Dict[str, Any]]:
        pairs = [(query, item["chunk"]["text"]) for item in retrieved]
        scores = self.model.predict(pairs)

        rescored = []
        for item, score in zip(retrieved, scores):
            rescored.append({
                "score": float(score),
                "chunk": item["chunk"],
            })
        rescored.sort(key=lambda x: x["score"], reverse=True)
        return rescored[:top_k]

reranker = OptionalReranker(config.reranker_model_name) if config.use_reranker else None
print("Reranker ativado." if reranker else "Reranker desativado.")

Reranker desativado.


## 9) Retrieval e debug

In [11]:
def retrieve_context(
    query: str,
    vector_index: VectorIndex,
    config: RAGConfig,
    metadata_filter: Optional[Dict[str, Any]] = None,
) -> List[Dict[str, Any]]:
    retrieved = vector_index.search(query, top_k=config.top_k, metadata_filter=metadata_filter)
    if config.use_reranker and reranker is not None and retrieved:
        retrieved = reranker.rerank(query, retrieved, top_k=config.top_k_rerank)
    else:
        retrieved = retrieved[:config.top_k_rerank]
    return retrieved

def preview_retrieval(query: str, metadata_filter: Optional[Dict[str, Any]] = None) -> pd.DataFrame:
    rows = []
    retrieved = retrieve_context(query, vector_index, config, metadata_filter)
    for item in retrieved:
        c = item["chunk"]
        rows.append({
            "score": round(item["score"], 4),
            "title": c["title"],
            "source_path": c["source_path"],
            "source_type": c["source_type"],
            "chunk_index": c["metadata"].get("chunk_index"),
            "page": c["metadata"].get("page"),
            "text_preview": c["text"][:240].replace("\n", " "),
        })
    return pd.DataFrame(rows)

# Exemplo:
preview_retrieval("Qual a melhor skill de pilotagem?")

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.55it/s]


,score,title,source_path,source_type,chunk_index,page,text_preview
0,0.3438,starfield_skills | page 2,rag_data/starfield_skills.pdf,pdf_page,0,2,Overview Skills are grouped into five categor...
1,0.3245,starfield_skills | page 4,rag_data/starfield_skills.pdf,pdf_page,0,4,Expert Marksmanship · Particle Beams · Rapid R...
2,0.3227,starfield_skills | page 1,rag_data/starfield_skills.pdf,pdf_page,0,1,STARFIELD SKILLS Skills are the primary progr...
3,0.3116,starfield_skills | page 3,rag_data/starfield_skills.pdf,pdf_page,0,3,Expert Cellular Regeneration · Decontamination...


## 10) Prompt grounding

In [12]:
SYSTEM_PROMPT = '''
Você é um assistente de RAG para ajudar com habilidades de Starfield.
Responda apenas com base no contexto fornecido.
Se a resposta não estiver claramente suportada pelo contexto, diga:
"Não encontrei informação suficiente no contexto fornecido."

Regras:
1. Não invente fatos.
2. Seja direto e claro.
3. Sempre cite as fontes usadas no final.
4. Se houver conflito entre fontes, mencione o conflito.
5. Se a pergunta pedir procedimento, responda em passos objetivos.
'''.strip()

def build_context_block(retrieved: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, item in enumerate(retrieved, start=1):
        c = item["chunk"]
        meta = c["metadata"]
        source_ref = f"{Path(c['source_path']).name}"
        if meta.get("page"):
            source_ref += f" | página {meta['page']}"
        source_ref += f" | chunk {meta.get('chunk_index')}"
        blocks.append(
            f"[Fonte {i}]\n"
            f"Título: {c['title']}\n"
            f"Origem: {source_ref}\n"
            f"Conteúdo:\n{c['text']}"
        )
    return "\n\n".join(blocks)

def build_user_prompt(question: str, retrieved: List[Dict[str, Any]]) -> str:
    context_block = build_context_block(retrieved)
    return f'''
Contexto recuperado:
{context_block}

Pergunta do usuário:
{question}

Formato esperado:
- Resposta objetiva
- Se útil, use bullets curtos
- Finalize com: "Fontes usadas:"
'''.strip()

## 11) Teste de conexão com Ollama

In [14]:
import requests

def test_ollama_connection(config: RAGConfig) -> Dict[str, Any]:
    resp = requests.get(f"{config.ollama_base_url}/api/tags", timeout=30)
    resp.raise_for_status()
    data = resp.json()

    models = [m["name"] for m in data.get("models", [])]
    print("Modelos disponíveis no Ollama:")
    for model in models:
        print("-", model)

    if not any(config.llm_model in m for m in models):
        print(f"\n[WARN] O modelo '{config.llm_model}' não apareceu na lista.")
        print("Talvez você precise rodar: ollama pull", config.llm_model)

    return data

test_ollama_connection(config)

Modelos disponíveis no Ollama:
- llama3.1:latest
- mxbai-embed-large:latest
- llama3:latest


{'models': [{'name': 'llama3.1:latest',
   'model': 'llama3.1:latest',
   'modified_at': '2026-04-07T18:29:07.099391519-03:00',
   'size': 4920753328,
   'digest': '46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'llama',
    'families': ['llama'],
    'parameter_size': '8.0B',
    'quantization_level': 'Q4_K_M'}},
  {'name': 'mxbai-embed-large:latest',
   'model': 'mxbai-embed-large:latest',
   'modified_at': '2026-04-07T18:17:50.949086034-03:00',
   'size': 669615493,
   'digest': '468836162de7f81e041c43663fedbbba921dcea9b9fefea135685a39b2d83dd8',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'bert',
    'families': ['bert'],
    'parameter_size': '334M',
    'quantization_level': 'F16'}},
  {'name': 'llama3:latest',
   'model': 'llama3:latest',
   'modified_at': '2026-04-07T18:17:27.056013967-03:00',
   'size': 4661224676,
   'digest': '365c0bd3c000a25d28ddbf732fe1c6add

## 12) Geração via Ollama

In [15]:
def generate_answer_llm(
    question: str,
    retrieved: List[Dict[str, Any]],
    config: RAGConfig,
) -> str:
    import requests

    user_prompt = build_user_prompt(question, retrieved)

    payload = {
        "model": config.llm_model,
        "prompt": f"{SYSTEM_PROMPT}\n\n{user_prompt}",
        "stream": False,
        "options": {
            "temperature": config.temperature,
            "num_predict": config.max_output_tokens,
        }
    }

    response = requests.post(
        f"{config.ollama_base_url}/api/generate",
        json=payload,
        timeout=300,
    )
    response.raise_for_status()
    data = response.json()
    return data["response"]

## 13) Pipeline final

In [17]:
def ask_rag(
    question: str,
    vector_index: VectorIndex,
    config: RAGConfig,
    metadata_filter: Optional[Dict[str, Any]] = None,
    debug: bool = True,
) -> Dict[str, Any]:
    t0 = time.time()
    retrieved = retrieve_context(question, vector_index, config, metadata_filter)
    retrieval_time = time.time() - t0

    answer = generate_answer_llm(question, retrieved, config)
    total_time = time.time() - t0

    result = {
        "question": question,
        "answer": answer,
        "retrieved": retrieved,
        "metrics": {
            "retrieval_time_sec": round(retrieval_time, 3),
            "total_time_sec": round(total_time, 3),
            "num_context_chunks": len(retrieved),
        }
    }

    if debug:
        print("=== MÉTRICAS ===")
        print(result["metrics"])

        print("\n=== CONTEXTO RECUPERADO ===")
        for i, item in enumerate(retrieved, start=1):
            c = item["chunk"]
            print(f"\n[{i}] score={item['score']:.4f} | {c['title']} | {c['source_path']}")
            print(c["text"][:400].replace("\n", " "))
            print("-" * 90)

        print("\n=== RESPOSTA ===")
        print(answer)

    return result

# Exemplo:
result = ask_rag("Cite as skills de pilotagem?", vector_index, config)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.85it/s]


=== MÉTRICAS ===
{'retrieval_time_sec': 0.355, 'total_time_sec': 7.915, 'num_context_chunks': 4}

=== CONTEXTO RECUPERADO ===

[1] score=0.4774 | starfield_skills | page 4 | rag_data/starfield_skills.pdf
Expert Marksmanship · Particle Beams · Rapid Reloading · Sniper Certification · Targeting  Master Armor Penetration · Crippling · Sharpshooting  Science  Skill Tier  Skills  Novice Astrodynamics · Geology · Medicine · Research Methods · Surveying  Advanced  Botany · Scanning · Spacesuit Design · Weapon Engineering · Zoology  Expert Astrophysics · Chemistry · Outpost Engineering  Master Aneutronic 
------------------------------------------------------------------------------------------

[2] score=0.4413 | starfield_skills | page 3 | rag_data/starfield_skills.pdf
Expert Cellular Regeneration · Decontamination · Martial Arts  Master Concealment · Neurostrikes · Rejuvenation  Social  Skill Tier  Skills  Novice Commerce · Gastronomy · Persuasion · Scavenging · Theft  Advanced  Deception ·

## 14) Tabela de fontes

In [18]:
def get_sources_table(result: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for i, item in enumerate(result["retrieved"], start=1):
        c = item["chunk"]
        rows.append({
            "rank": i,
            "score": round(item["score"], 4),
            "title": c["title"],
            "source_path": c["source_path"],
            "source_type": c["source_type"],
            "page": c["metadata"].get("page"),
            "chunk_index": c["metadata"].get("chunk_index"),
            "text_preview": c["text"][:250].replace("\n", " "),
        })
    return pd.DataFrame(rows)

## 15) Dataset fake para teste

In [ ]:
def create_demo_dataset(base_dir: str = "./rag_data_demo") -> str:
    base = ensure_dir(base_dir)

    (base / "policy.md").write_text(
        '''
# Política de Reembolso

Reembolsos podem ser solicitados em até 7 dias corridos após a cobrança.
Após esse prazo, o caso deve ser analisado manualmente pelo time financeiro.
Chargebacks devem ser escalados para o time de risco.
'''.strip(),
        encoding="utf-8"
    )

    pd.DataFrame([
        {"id": 1, "category": "payment_error", "description": "Cartão recusado por saldo insuficiente."},
        {"id": 2, "category": "risk", "description": "Chargeback aberto pelo emissor do cartão."},
        {"id": 3, "category": "support", "description": "Cliente deseja cancelar a assinatura."},
    ]).to_csv(base / "tickets.csv", index=False)

    return str(base)

demo_dir = create_demo_dataset()
demo_dir

In [ ]:
demo_config = RAGConfig(data_dir=demo_dir, index_dir="./rag_index_demo", use_reranker=False)
demo_docs = load_documents(demo_config)
demo_chunks = build_chunks(demo_docs, demo_config)

demo_index = VectorIndex(demo_config)
demo_index.build(demo_chunks)

preview_retrieval("em quanto tempo o cliente pode pedir reembolso?")

## 16) Teste completo com o dataset fake

Descomente quando o Ollama estiver rodando e com o modelo baixado.

In [ ]:
# test_ollama_connection(demo_config)
# result = ask_rag("Em quanto tempo o cliente pode pedir reembolso?", demo_index, demo_config)
# print(result["answer"])
# get_sources_table(result)

## 17) Como adaptar para seus dados

### Docs em pasta
Jogue os arquivos em `rag_data/`.

### CSV de tickets / erros
Garanta uma coluna como:
- `description`
- `message`
- `text`
- `content`

Se o nome for outro:
```python
documents = load_documents(
    config,
    csv_text_overrides={"seu_arquivo.csv": "nome_da_coluna"}
)
```

### Filtro por arquivo
```python
ask_rag(
    "qual a regra de reembolso?",
    vector_index,
    config,
    metadata_filter={"file_name": "policy.md"}
)
```

## 18) Checklist rápido

- Ollama está rodando?
- modelo foi baixado?
- pasta `rag_data/` existe?
- `numpy` e `faiss` estão compatíveis?
- retrieval está trazendo chunks bons?
- só depois disso culpe o modelo 😎